In [1]:
import os, random, numpy as np, pandas as pd, GPy, warnings

warnings.filterwarnings("ignore")
GPy.util.linalg.jitchol_MAXTRIES = 20

ROOT_FOLDER = r"C:/Users/sysdp/OneDrive - 島根大学/Desktop/Research/FPGA DATA/Raw-data_exahaustive"
OUT_FOLDER =  r"C:/Users/sysdp/OneDrive - 島根大学/Desktop/Research/MY PAPERS/hybrid_svss_asde/sde_svr"

FPGA_START, FPGA_END = 1, 50
NUM_PATHS = 32
ALPHA, BETA = 2, 2
RATIO = 0.10
SEED = 42
CALIB_FPGA = 1


from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVR


def fit_predict(Xtr, Ytr, Xte):
    sx, sy = StandardScaler().fit(Xtr), StandardScaler().fit(Ytr.reshape(-1, 1))
    grid = {"C": [0.1, 1, 10, 100], "gamma": [0.1, 1, 10], "epsilon": [0.01, 0.1]}
    g = GridSearchCV(SVR(), grid, cv=3).fit(sx.transform(Xtr), sy.transform(Ytr.reshape(-1, 1)).ravel())
    pred = g.predict(sx.transform(Xte)).reshape(-1, 1)
    return sy.inverse_transform(pred).ravel(), np.full(len(Xte), np.nan)


def load_dat(fp):
    grid = np.loadtxt(fp, delimiter=",")
    rows, cols = grid.shape
    rec = [
        (x, y, grid[y, x])
        for y in range(rows)
        for x in range(cols)
        if grid[y, x] != 0.0
    ]
    df = pd.DataFrame(rec, columns=["X", "Y", "Value"])
    df = df[np.abs(df.Value - df.Value.median()) <= 3 * df.Value.std()]
    return df.reset_index(drop=True)


def sde_select(xs, ys, n_target, rng):
    order = list(range(len(xs)))
    rng.shuffle(order)
    selected = [order[0]]
    leftover = []
    for idx in order[1:]:
        if len(selected) >= n_target:
            break
        if all(
            abs(xs[idx] - xs[s]) > ALPHA or abs(ys[idx] - ys[s]) > BETA
            for s in selected
        ):
            selected.append(idx)
        else:
            leftover.append(idx)
    if len(selected) < n_target:
        selected += leftover[: n_target - len(selected)]
    return selected


calib_positions = {}


def get_calib_positions(path):
    if path not in calib_positions:
        df = load_dat(f"{ROOT_FOLDER}/FPGA-{CALIB_FPGA}/data{path}.dat")
        n_train = int(round(RATIO * len(df)))
        sel = sde_select(df.X.values, df.Y.values, n_train, random.Random(SEED))
        calib_positions[path] = set(map(tuple, df.loc[sel, ["X", "Y"]].values))
    return calib_positions[path]


def run_path(fpga, path):
    f1 = f"{OUT_FOLDER}/FPGA{fpga}/path{path}_sde_svr.csv"
    f2 = f"{OUT_FOLDER}/FPGA{fpga}/fpga{fpga}_path{path}_sde_svr.csv"
    if os.path.exists(f1) or os.path.exists(f2):
        return
    df = load_dat(f"{ROOT_FOLDER}/FPGA-{fpga}/data{path}.dat")
    pos_set = get_calib_positions(path)
    train_idx = df[df.apply(lambda r: (r.X, r.Y) in pos_set, axis=1)].index.tolist()
    test_idx = [i for i in df.index if i not in train_idx]

    X = df[["X", "Y"]].values
    Xtr, ytr = X[train_idx], df.loc[train_idx, "Value"].values.reshape(-1, 1)
    Xte, yte = X[test_idx], df.loc[test_idx, "Value"].values

    pred, var = fit_predict(Xtr, ytr, Xte)

    out = pd.DataFrame(
        {
            "X": Xte[:, 0],
            "Y": Xte[:, 1],
            "VALUE": yte,
            "PREDICTED VALUE": pred.flatten(),
            "VARIANCE": var.flatten(),
        }
    )
    fpga_folder = f"{OUT_FOLDER}/FPGA{fpga}"
    os.makedirs(fpga_folder, exist_ok=True)
    out.to_csv(f"{fpga_folder}/path{path}_sde_svr.csv", index=False)
    print(f"FPGA-{fpga} Path-{path}: train={len(train_idx)}/{len(df)} saved.")


def main():
    for fpga in range(FPGA_START, FPGA_END + 1):
        for path in range(1, NUM_PATHS + 1):
            run_path(fpga, path)


if __name__ == "__main__":
    main()

FPGA-1 Path-1: train=316/3155 saved.
FPGA-1 Path-2: train=317/3173 saved.
FPGA-1 Path-3: train=317/3173 saved.
FPGA-1 Path-4: train=317/3173 saved.
FPGA-1 Path-5: train=317/3173 saved.
FPGA-1 Path-6: train=317/3173 saved.
FPGA-1 Path-7: train=317/3173 saved.
FPGA-1 Path-8: train=317/3173 saved.
FPGA-1 Path-9: train=317/3173 saved.
FPGA-1 Path-10: train=317/3173 saved.
FPGA-1 Path-11: train=317/3173 saved.
FPGA-1 Path-12: train=317/3173 saved.
FPGA-1 Path-13: train=317/3173 saved.
FPGA-1 Path-14: train=317/3173 saved.
FPGA-1 Path-15: train=317/3173 saved.
FPGA-1 Path-16: train=317/3173 saved.
FPGA-1 Path-17: train=317/3173 saved.
FPGA-1 Path-18: train=317/3173 saved.
FPGA-1 Path-19: train=317/3173 saved.
FPGA-1 Path-20: train=317/3173 saved.
FPGA-1 Path-21: train=317/3173 saved.
FPGA-1 Path-22: train=317/3173 saved.
FPGA-1 Path-23: train=317/3173 saved.
FPGA-1 Path-24: train=317/3173 saved.
FPGA-1 Path-25: train=317/3173 saved.
FPGA-1 Path-26: train=317/3173 saved.
FPGA-1 Path-27: train

In [2]:
import os

OUT_FOLDER = r"C:/Users/sysdp/OneDrive - 島根大学/Desktop/Research/MY PAPERS/hybrid_svss_asde/sde_svr"


for fpga_dir in os.listdir(OUT_FOLDER):
    fpga_path = os.path.join(OUT_FOLDER, fpga_dir)
    n = fpga_dir.replace("FPGA", "")
    for f in os.listdir(fpga_path):
        if f.startswith("path") and f.endswith("_sde_svr.csv"):
            new_name = f"fpga{n}_{f}"
            os.rename(os.path.join(fpga_path, f), os.path.join(fpga_path, new_name))

In [3]:
import os, re, pandas as pd

OUT_FOLDER = r"C:/Users/sysdp/OneDrive - 島根大学/Desktop/Research/MY PAPERS/hybrid_svss_asde/sde_svr"

dfs = []
for fpga_dir in os.listdir(OUT_FOLDER):
    fpga_path = os.path.join(OUT_FOLDER, fpga_dir)
    for f in os.listdir(fpga_path):
        if f.endswith("_sde_svr.csv"):
            fpga, path = re.findall(r"fpga(\d+)_path(\d+)", f)[0]
            df = pd.read_csv(os.path.join(fpga_path, f))[
                ["X", "Y", "VALUE", "PREDICTED VALUE", "VARIANCE"]
            ]
            df["FPGA"], df["PATH"] = int(fpga), int(path)
            dfs.append(df)

combined = pd.concat(dfs, ignore_index=True)
combined = combined.sort_values(["FPGA", "PATH"]).reset_index(drop=True)
combined.to_csv(f"{OUT_FOLDER}/combined_sde_svr.csv", index=False)


In [4]:
import pandas as pd

OUT_FOLDER = r"C:/Users/sysdp/OneDrive - 島根大学/Desktop/Research/MY PAPERS/hybrid_svss_asde/sde_svr"

df = pd.read_csv(f"{OUT_FOLDER}/combined_sde_svr.csv")
df = df.sort_values(["FPGA", "PATH"]).reset_index(drop=True)
df.to_csv(f"{OUT_FOLDER}/combined_sde_svr.csv", index=False)

In [5]:
import numpy as np, pandas as pd

ROOT_FOLDER = "C:/Users/sysdp/OneDrive - 島根大学/Desktop/Research/FPGA DATA/Raw-data_exahaustive"
OUT_FOLDER = "C:/Users/sysdp/OneDrive - 島根大学/Desktop/Research/MY PAPERS/hybrid_svss_asde/sde_svr"
FPGA_START, FPGA_END = 1, 50
NUM_PATHS = 32


def load_dat(fp):
    grid = np.loadtxt(fp, delimiter=",")
    rows, cols = grid.shape
    vals = [grid[y, x] for y in range(rows) for x in range(cols) if grid[y, x] != 0.0]
    vals = pd.Series(vals)
    vals = vals[np.abs(vals - vals.median()) <= 3 * vals.std()]
    return vals


rows = []
for fpga in range(FPGA_START, FPGA_END + 1):
    for path in range(1, NUM_PATHS + 1):
        vals = load_dat(f"{ROOT_FOLDER}/FPGA-{fpga}/data{path}.dat")
        rows.append(
            {
                "FPGA": fpga,
                "PATH": path,
                "Max Value": vals.max(),
                "Min Value": vals.min(),
                "dspec": vals.max() - vals.min(),
            }
        )

dspec_df = pd.DataFrame(rows)

df = pd.read_csv(f"{OUT_FOLDER}/combined_sde_svr.csv")
df = df.merge(dspec_df, on=["FPGA", "PATH"], how="left")
df.to_csv(f"{OUT_FOLDER}/combined_sde_svr.csv", index=False)


In [6]:
import pandas as pd

OUT_FOLDER = "C:/Users/sysdp/OneDrive - 島根大学/Desktop/Research/MY PAPERS/hybrid_svss_asde/sde_svr"

df = pd.read_csv(f"{OUT_FOLDER}/combined_sde_svr.csv")
df["delta"] = (df["PREDICTED VALUE"] - df["VALUE"]) / df["dspec"]

per_path = (
    df.groupby(["FPGA", "PATH"])["delta"].apply(lambda x: x.abs().mean()).reset_index()
)
per_path.columns = ["FPGA", "PATH", "Delta"]
per_path.to_csv(f"{OUT_FOLDER}/sde_svr_delta_per_path.csv", index=False)

per_fpga = per_path.groupby("FPGA")["Delta"].mean().reset_index()
per_fpga.to_csv(f"{OUT_FOLDER}/sde_svr_delta_per_fpga.csv", index=False)

overall = per_fpga["Delta"].mean()
pd.DataFrame({"Overall Delta": [overall]}).to_csv(
    f"{OUT_FOLDER}/sde_svr_delta_overall.csv", index=False
)
print(f"Overall Delta: {overall:.6f}")

Overall Delta: 0.076953


In [ ]:
import os, numpy as np, pandas as pd

DATA_ROOT = r"C:/Users/sysdp/OneDrive - 島根大学/Desktop/Research/FPGA DATA/Raw-data_exahaustive"
TEST_ROOT = r"C:/Users/sysdp/OneDrive - 島根大学/Desktop/Research/MY PAPERS/hybrid_svss_asde/sde_svr/FPGA1"
OUT_ROOT = r"C:/Users/sysdp/OneDrive - 島根大学/Desktop/Research/MY PAPERS/hybrid_svss_asde/FPGA1T"

FPGA = 1
NUM_PATHS = 32

os.makedirs(OUT_ROOT, exist_ok=True)

for path in range(1, NUM_PATHS + 1):
    # Load full raw data
    grid = np.loadtxt(
        os.path.join(DATA_ROOT, f"FPGA-{FPGA}", f"data{path}.dat"), delimiter=","
    )
    rows, cols = grid.shape
    rec = [
        (x, y, grid[y, x])
        for y in range(rows)
        for x in range(cols)
        if grid[y, x] != 0.0
    ]
    full = pd.DataFrame(rec, columns=["X", "Y", "Value"])
    median, std = full.Value.median(), full.Value.std()
    full = full[np.abs(full.Value - median) <= 3 * std].reset_index(drop=True)

    # Load test points CSV
    test_file = os.path.join(TEST_ROOT, f"fpga{FPGA}_path{path}_sde_svr.csv")
    if not os.path.exists(test_file):
        print(f"Path-{path}: test file not found, skipping.")
        continue
    test = pd.read_csv(test_file)[["X", "Y"]]

    # Training points = full data minus test points
    test_set = set(zip(test.X, test.Y))
    train = full[~full.apply(lambda r: (r.X, r.Y) in test_set, axis=1)]

    train.to_csv(
        os.path.join(OUT_ROOT, f"fpga{FPGA}_path{path}_train_sde_svr.csv"), index=False
    )
    print(f"Path-{path}: {len(train)} training points saved.")
